## HDC(QSVM)

In [1]:
import numpy as np
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.metrics import accuracy_score
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
np.random.seed(42)
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

sampler = Sampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

start_train_time = time.time()

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\586584604.py:55: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 85.29%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 82.35%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 1.4937 seconds
Hybrid Clasical-Quantum HDC(QSVM) Inference Time: 1.4942 seconds
Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: 0.0036 MB


## IBM AerSimulator (HDC(QSVM))

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.metrics import accuracy_score
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import RealAmplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_gate_map, plot_circuit_layout, plot_distribution
from qiskit.circuit import ParameterVector

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

#num_qubits = 50

#service = QiskitRuntimeService()
#backend = service.backend("ibm_osaka")

#fake_backend = AerSimulator.from_backend(backend)

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
np.random.seed(42)
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

aer_sim = AerSimulator()
sampler = Sampler(backend=aer_sim)

fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

start_train_time = time.time()

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\2934466661.py:86: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=aer_sim)


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 82.35%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 88.24%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 3.0688 seconds
Hybrid Clasical-Quantum HDC(QSVM) Inference Time: 2.9719 seconds
Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: 0.0036 MB


## IBM IBM Kyto Hardware HDC(QSVM)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.metrics import accuracy_score
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import RealAmplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_gate_map, plot_circuit_layout, plot_distribution
from qiskit.circuit import ParameterVector

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

service = QiskitRuntimeService()
backend = service.backend("ibm_kyoto")

fake_backend = AerSimulator.from_backend(backend)

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
np.random.seed(42)
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

#aer_sim = AerSimulator()
sampler = Sampler(backend=fake_backend)

fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

start_train_time = time.time()

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\877956844.py:42: DeprecationWarning: The class ``qiskit.providers.models.backendconfiguration.QasmBackendConfiguration`` is deprecated as of qiskit 1.2. It will be removed in the 2.0 release. The models in ``qiskit.providers.models`` are part of the deprecated `BackendV1` workflow and no longer necessary for `BackendV2`. If a user workflow requires these representations it likely relies on deprecated functionality and should be updated to use `BackendV2`.
  fake_backend = AerSimulator.from_backend(backend)


Generating random projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\877956844.py:84: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=fake_backend)


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 82.35%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 88.24%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 7.1657 seconds
Hybrid Clasical-Quantum HDC(QSVM) Inference Time: 5.2126 seconds
Hybrid Clasical-Quantum HDC(QSVM) Model Memory Required: 0.0036 MB


## HDC(QNN)

In [4]:
import numpy as np
import sys
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap, PauliFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2  # Match the number of features in the original data

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Project training and test data to hyperdimensional space
X_train_proj = project_data(X_train, proj)
X_test_proj = project_data(X_test, proj)

# Define Quantum Circuit with a higher number of reps
qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increase reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
)

# Use L-BFGS-B optimizer for better performance with more complex models
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...
Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 67.65%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 76.47%
Hybrid Classical-Quantum HDC(QNN) Training Time: 16.0991 seconds
Hybrid Clasical-Quantum HDC(QNN) Inference Time: 0.0315 seconds
Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: 0.0010 MB


## IBM AerSimulator (HDC(QNN))

In [6]:
import numpy as np
import sys
import time
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import RealAmplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_gate_map, plot_circuit_layout, plot_distribution
from qiskit.circuit import ParameterVector
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap, PauliFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

#num_qubits = 50

#service = QiskitRuntimeService()
#backend = service.backend("ibm_osaka")

#fake_backend = AerSimulator.from_backend(backend)

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2  # Match the number of features in the original data

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Project training and test data to hyperdimensional space
X_train_proj = project_data(X_train, proj)
X_test_proj = project_data(X_test, proj)

# Define Quantum Circuit with a higher number of reps
qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increase reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

aer_sim = AerSimulator()
sampler = Sampler(backend=aer_sim)

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
    sampler = sampler
)

# Use L-BFGS-B optimizer for better performance with more complex models
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating random projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\306504181.py:88: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=aer_sim)


Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 79.41%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 41.18%
Hybrid Classical-Quantum HDC(QNN) Training Time: 47.9462 seconds
Hybrid Clasical-Quantum HDC(QNN) Inference Time: 0.1655 seconds
Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: 0.0010 MB


## IBM Kyto Hardware HDC(QNN)

In [8]:
import numpy as np
import sys
import time
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import RealAmplitudes
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_gate_map, plot_circuit_layout, plot_distribution
from qiskit.circuit import ParameterVector
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap, PauliFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

service = QiskitRuntimeService()
backend = service.backend("ibm_kyoto")

fake_backend = AerSimulator.from_backend(backend)

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2  # Match the number of features in the original data

# Create a random projection matrix
print("Generating random projection...")
proj = np.random.randn(D, X_train.shape[1])

def project_data(data, proj):
    return np.dot(data, proj.T)

# Project training and test data to hyperdimensional space
X_train_proj = project_data(X_train, proj)
X_test_proj = project_data(X_test, proj)

# Define Quantum Circuit with a higher number of reps
qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increase reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

#aer_sim = AerSimulator()
sampler = Sampler(fake_backend)

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
    sampler = sampler
)

# Use L-BFGS-B optimizer for better performance with more complex models
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Clasical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\414236973.py:40: DeprecationWarning: The class ``qiskit.providers.models.backendconfiguration.QasmBackendConfiguration`` is deprecated as of qiskit 1.2. It will be removed in the 2.0 release. The models in ``qiskit.providers.models`` are part of the deprecated `BackendV1` workflow and no longer necessary for `BackendV2`. If a user workflow requires these representations it likely relies on deprecated functionality and should be updated to use `BackendV2`.
  fake_backend = AerSimulator.from_backend(backend)


Generating random projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_6280\414236973.py:86: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(fake_backend)


Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 79.41%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 58.82%
Hybrid Classical-Quantum HDC(QNN) Training Time: 148.9496 seconds
Hybrid Clasical-Quantum HDC(QNN) Inference Time: 1.8793 seconds
Hybrid Clasical-Quantum HDC(QNN) Model Memory Required: 0.0010 MB
